In [1]:
import boto3
import io
import glob
import logging
import math
import os
import requests
import s3fs
import subprocess
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv
from pathlib import Path

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [4]:
# ============================================================
#       Args
# ============================================================

# File intake
FILE_GENE_LIST = f"{REPO_ROOT}/results/2026-10-07_for_c_marshal.tsv"
FILE_MARSHALL_AI = f"{REPO_ROOT}/data_sync/c_marshal_ai_panel_genes.xlsx"

# Dirs
OUT_DIR = f"{REPO_ROOT}/results"

In [9]:
# ============================================================
#       In -- gene list
# ============================================================

df_gene_list = pl.read_csv(
    FILE_GENE_LIST,
    separator="\t",
    has_header=True,
    infer_schema_length=None,
    null_values="NA"
)

df_gene_list

defense_uniprot_ac,defense_foldseek_repId,defense_taxId,human_uniprot_ac,human_uniprot_reviewed,human_protein_names,human_gene_names,human_length_AAs,human_gene_name_primary,uniprot_ref_proteome_chr,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (molecular function),Gene Ontology (GO),Gene Ontology IDs,FunFam,CDD,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,defense_type,defense_subtype,defense_species,defense_sys_id,defense_sys_beg,defense_sys_end,defense_duplicate_count,defense_rank,human_domains,defense_gene_name,tm_score_human,tm_score_bacteria,rmsd,…,human_aligned_start,human_aligned_end,bact_aligned_start,bact_aligned_end,evalue_foldseek,griid_gene,bacterial_length,bacterial_domains,in_conserved_module,module_count,module_partner_genes,plddt_human_mean,plddt_bacteria_mean,tm_pvalue_human,tm_pvalue_bacteria,tm_pvalue,q_value,structure_source_human,structure_source_bacteria,alignment_status,immune_relevant,composite_score,alphafold_model_version,run_date,human_gene_in_IUIS,defense_AA_sequence,human_AA_sequence,blastp_pident,blastp_length,blastp_mismatch,blastp_gapopen,blastp_qstart,blastp_qend,blastp_sstart,blastp_send,blastp_evalue,blastp_bitscore
str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,f64,f64,f64,…,i64,i64,i64,i64,f64,bool,i64,str,bool,i64,str,f64,f64,f64,f64,f64,f64,str,str,str,bool,f64,str,str,bool,str,str,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""Q47ZY9""","""A0A2E5CU08""",167879,"""Q8IZH9""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,177,null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.30.113…","""cd00642;""","""1.10.286.10;3.30.1130.10;""","""IPR043133;IPR043134;IPR001474;…","""PTHR11109:SF11;PTHR11109;""","""PF01227;""","""PS00859;PS00860;""",null,"""NP_001019195.1;""","""Dpd""","""Dpd""","""Colwellia psychrerythraea""","""GCF_000012325_NC_003910_Dpd_1""","""GCF_000012325.1_NC_003910_0264…","""GCF_000012325.1_NC_003910_0265…",1,202,"""DOMAIN 2..176; /note=""GTP cycl…","""Dpd__FolE""",0.991,0.964,0.51,…,1,176,6,181,0.0,false,182,"""DOMAIN 3..180; /note=""GTP cycl…",false,0,null,95.53,97.12,8.8818e-16,2.5535e-15,2.5535e-15,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9962,"""model_v6""","""""",null,"""MTIENNYREILATIGEDINRGGLLDTPKRA…","""AYSSILSSLGENPQRQGLLKTPWRAASAMQ…",56.0,175,77,0,7,181,2,176,7.3000e-82,227.0
"""A0A5B9E1W6""","""A0A7C2NS94""",582702,"""Q5QP19""","""unreviewed""","""Cysteine desulfurase (EC 2.8.1…","""NFS1 hCG_38220""",397,"""NFS1""",null,"""[2Fe-2S] cluster assembly [GO:…","""cytosol [GO:0005829]; mitochon…","""cysteine desulfurase activity …","""cytosol [GO:0005829]; mitochon…","""GO:0005654; GO:0005829; GO:000…","""3.40.640.10:FF:000003;3.90.115…",null,"""3.90.1150.10;3.40.640.10;""","""IPR000192;IPR020578;IPR010240;…","""PTHR11601:SF34;PTHR11601;""","""PF00266;""","""PS00595;""",null,null,"""Dnd""","""Dnd_ABCDE""","""Rhodanobacter glycinis""","""GCF_008000795_NZ_CP042807_Dnd_…","""GCF_008000795.1_NZ_CP042807_02…","""GCF_008000795.1_NZ_CP042807_02…",1,202,"""DOMAIN 2..361; /note=""Aminotra…","""Dnd__DndA""",0.9237,0.9496,1.3,…,1,376,9,386,0.0,false,386,"""DOMAIN 7..369; /note=""Aminotra…",false,0,null,94.52,95.11,1.3656e-14,4.6629e-15,1.3656e-14,1.7883e-9,"""afdb_v6""","""afdb_v6""","""ok""",false,0.9863,"""model_v6""","""""",false,"""MSDGRPIYLDHNATTPLDPTVFEAMRPYFL…","""MDVQATTPLDPRVLDAMLPYLINYYGNPHS…",44.18,378,208,3,9,385,1,376,5.3700e-114,326.0
"""Q47ZY9""","""A0A2E5CU08""",167879,"""Q8IZH8""","""unreviewed""","""GTP cyclohydrolase 1 (EC 3.5.4…",null,158,null,null,"""regulation of blood pressure […","""cytoplasm [GO:0005737]; nucleu…","""GTP binding [GO:0005525]; GTP …","""cytoplasm [GO:0005737]; nucleu…","""GO:0003934; GO:0005525; GO:000…","""1.10.286.10:FF:000003;3.

In [10]:
# ============================================================
#       In -- C Marshall panel
# ============================================================

df_marshall_panel = pl.concat([
    pl.read_excel(
        FILE_MARSHALL_AI,
        sheet_name=sheet
    ).with_columns(
        pl.lit(sheet).alias("source_sheet")
    )
    for sheet in ["RFS", "HLH", "AI", "AG"]
])

df_marshall_panel

Subpanel,snpEff_gene,snpEff_refseq,source_sheet
str,str,str,str
"""RFS""","""MEFV""","""NM_000243.2""","""RFS"""
"""RFS""","""MVK""","""NM_000431.3""","""RFS"""
"""RFS""","""NLRP12""","""NM_144687.3""","""RFS"""
"""RFS""","""NLRP3""","""NM_004895.4""","""RFS"""
"""RFS""","""TNFRSF1A""","""NM_001065.3""","""RFS"""
…,…,…,…
"""AG""","""RNASEH2A""","""NM_006397.2""","""AG"""
"""AG""","""RNASEH2B""","""NM_024570.3""","""AG"""
"""AG""","""RNASEH2C""","""NM_032193.3""","""AG"""
